# Pre-process des données

### Importation des données et nettoyage initial

In [15]:
import pandas as pd
import numpy as np

data = pd.ExcelFile("data_CRIR_anon.xlsx")

data_P = pd.read_excel(data, 'P')
data_S = pd.read_excel(data, 'S')
data_EF = pd.read_excel(data, 'EF')
data_T = pd.read_excel(data, 'T')

##on enlève la première ligne qui correspond à des commentaires
data_P.drop(0, inplace=True)
data_S.drop(0, inplace=True)
data_EF.drop(0, inplace=True)
data_T.drop(0, inplace=True)

## On enlève les lignes avec P6_reel = 1, car elles ne correspondent à rien dans les autres tableaux, 
# et on les fait correspondre à P6_target pour pouvoir join les tableaux
data_EF_clean = data_EF.drop(data_EF[data_EF['P6_reel']==1.00].index)
data_EF_clean['round_P6']= data_EF_clean['P6_reel'].round(0)

### Création du dataframe global

In [16]:
## On calcule les moyennes et écart-type de S1 par essai.
data_S['S1'] = pd.to_numeric(data_S['S1'], errors='coerce')
data_S_m = data_S.groupby('essai', as_index=False).agg(
    S_moy=('S1', 'mean'),
    sigma_S=('S1', 'std')
)
data_S_m['essai'] = data_S_m['essai'].astype(int)
data_S_m.head()

,essai,S_moy,sigma_S
0,390310,2.763667,0.414240
1,392291,2.992202,0.551089
2,392292,3.296931,0.549037
3,392293,2.595811,0.471396
4,392294,3.354443,0.585051


In [17]:
df= pd.merge(data_EF_clean, data_P, how='left', left_on=['essai','round_P6'] , right_on=['essai','P6_target'])
df[['E2.1', 'E2.2', 'E2.3', 'E2.4']] = df[['E2.1', 'E2.2', 'E2.3', 'E2.4']].astype(float)
df["moyenne_E2"] = df[["E2.1", "E2.2", "E2.3", "E2.4"]].mean(axis=1)
df= pd.merge(df, data_S_m, how='left', left_on=['essai'] , right_on=['essai'])
df.to_csv('mes_donnees.csv', sep=';', index=False, encoding='utf-8-sig')
df.head(25)

,essai,P6_reel,E1,V,E2,P1_reel,Lq1_reel,Lq2_reel,Lq_reel,Lq_target_x,...,P2,Lt,Lq_target_y,P3,P4,P5,P6_target,moyenne_E2,S_moy,sigma_S
0,392291.0,3.20,86.5,0.0,1103,15.12,4.3,4.2,4.3,3.7,...,87.0,R,0.04,16.0,14.712644,48,3.0,NaN,2.992202,0.551089
1,392291.0,3.20,84.5,0.0,876,15.12,3.8,3.9,3.8,3.7,...,87.0,R,0.04,16.0,14.712644,48,3.0,NaN,2.992202,0.551089
2,392291.0,3.20,87.9,0.0,982,15.12,3.9,3.8,3.9,3.7,...,87.0,R,0.04,16.0,14.712644,48,3.0,NaN,2.992202,0.551089
3,392291.0,3.20,84.6,0.0,905,15.12,4.1,3.9,4,3.7,...,87.0,R,0.04,16.0,14.712644,48,3.0,NaN,2.992202,0.551089
4,392291.0,3.20,88.4,0.0,1087,15.12,4.3,4.2,4.3,3.7,...,87.0,R,0.04,16.0,14.712644,48,3.0,41.50,2.992202,0.551089
5,392291.0,3.20,84.9,0.0,874,15.12,3.8,3.9,3.8,3.7,...,87.0,R,0.04,16.0,14.712644,48,3.0,52.75,2.992202,0.551089
6,392291.0,3.20,85.5,0.0,883,15.12,3.9,3.8,3.9,3.7,...,87.0,R,0.04,16.0,14.712644,48,3.0,51.75,2.992202,0.551089
7,392291.0,3.20,84.6,0.0,900,15.12,4.1,3.9,4,3.7,...,87.0,R,0.04,16.0,14.712644,48,3.0,44.00,2.992202,0.551089
8,392291.0,4.92,86.3,0.0,1079,15.12,4.5,4.1,4.3,3.7,...,87.0,R,0.04,16.0,14.712644,80,5.0,NaN,2.992202,0.551089
9,392291.0,4.92,84.1,0.0,874,15.12,4.1,3.7,3.9,3.7,...,87.0,R,0.04,16.0,14.712644,80,5.0,NaN,2.992202,0.551089


### Normalisation des variables numériques et binairisation des variables textuelles.

In [18]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
df_normalize = df.copy()

df_normalize['F1'] = df_normalize['F1'].map({'Cold': 0, 'FME': 1})
df_normalize['Lt'] = df_normalize['Lt'].map({'R': 0, 'G': 1})

colonnes_a_enlever = ['date', 'condition', 'essai', 'P6_reel']
masque = ~df_normalize.columns.isin(colonnes_a_enlever)

df_normalize.loc[:, masque] = scaler.fit_transform(df_normalize.loc[:, masque])

df_normalize.head()

,essai,P6_reel,E1,V,E2,P1_reel,Lq1_reel,Lq2_reel,Lq_reel,Lq_target_x,...,P2,Lt,Lq_target_y,P3,P4,P5,P6_target,moyenne_E2,S_moy,sigma_S
0,392291.0,3.2,0.508681,0.0,0.762007,0.531993,0.140432,0.086909,0.139073,0.105263,...,0.0,0.0,0.0,0.5,0.646018,0.0,0.0,NaN,0.458702,0.137857
1,392291.0,3.2,0.480526,0.0,0.599283,0.531993,0.063272,0.053905,0.056291,0.105263,...,0.0,0.0,0.0,0.5,0.646018,0.0,0.0,NaN,0.458702,0.137857
2,392291.0,3.2,0.52839,0.0,0.675269,0.531993,0.078704,0.042904,0.072848,0.105263,...,0.0,0.0,0.0,0.5,0.646018,0.0,0.0,NaN,0.458702,0.137857
3,392291.0,3.2,0.481933,0.0,0.620072,0.531993,0.109568,0.053905,0.089404,0.105263,...,0.0,0.0,0.0,0.5,0.646018,0.0,0.0,NaN,0.458702,0.137857
4,392291.0,3.2,0.535429,0.0,0.750538,0.531993,0.140432,0.086909,0.139073,0.105263,...,0.0,0.0,0.0,0.5,0.646018,0.0,0.0,0.059949,0.458702,0.137857


In [19]:
df_normalize.to_csv('mes_donnees_normalisées.csv', sep=';', index=False, encoding='utf-8-sig')

In [20]:
#Régression linéaire en supprimant les lignes NA
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

colonnes_utiles = ["P1_reel", "P2", "Lt", "Lq_reel", "P4", "P5", "V", "E2", "P6_reel", "E1"]
masque = df_normalize.columns.isin(colonnes_utiles)
df2 = df_normalize.loc[:, masque]

df_normalizedrop = df2.dropna()
df_normalizedrop.head()
print(len(df_normalizedrop))

X = df_normalizedrop.drop(columns = ["E1"])
Y = df_normalizedrop["E1"]

print(X,Y)

X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size = 0.2)

modele = LinearRegression()
modele.fit(X_train, y_train)

test = modele.predict(X_test)

erreur = mean_absolute_error(y_test, test)

print(f"Le modèle se trompe en moyenne de : {erreur}")

556
     P6_reel    V        E2   P1_reel   Lq_reel   P2   Lt        P4      P5
0       3.20  0.0  0.762007  0.531993  0.139073  0.0  0.0  0.646018     0.0
1       3.20  0.0  0.599283  0.531993  0.056291  0.0  0.0  0.646018     0.0
2       3.20  0.0  0.675269  0.531993  0.072848  0.0  0.0  0.646018     0.0
3       3.20  0.0  0.620072  0.531993  0.089404  0.0  0.0  0.646018     0.0
4       3.20  0.0  0.750538  0.531993  0.139073  0.0  0.0  0.646018     0.0
..       ...  ...       ...       ...       ...  ...  ...       ...     ...
699     5.12  0.0  0.682437  0.957952  0.701987  0.0  1.0       1.0  0.8125
700     5.12  0.0  0.863082  0.957952  0.668874  0.0  1.0       1.0  0.8125
701     5.12  0.0  0.832258  0.957952  0.602649  0.0  1.0       1.0  0.8125
702     5.12  0.0  0.894624  0.957952  0.586093  0.0  1.0       1.0  0.8125
703     5.12  0.0  0.687455  0.957952  0.701987  0.0  1.0       1.0  0.8125

[556 rows x 9 columns] 0      0.508681
1      0.480526
2       0.52839
3      0.481

In [21]:
#Régression linéaire N°2 en supprimant les lignes NA (Calcul des colonnes puis moyenne)
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

colonnes_utiles = ["P1_reel", "P2", "Lt", "Lq_reel", "P4", "P5", "V", "E2", "P6_reel", "E1", 'E2.1']
masque = df_normalize.columns.isin(colonnes_utiles)
df2 = df_normalize.loc[:, masque]

df_normalizedrop = df2.dropna()
df_normalizedrop.head()
print(len(df_normalizedrop))

X1 = df_normalizedrop.drop(columns = ['E2.1'])
Y1 = df_normalizedrop['E2.1']

X1_train, X1_test, y1_train, y1_test = train_test_split(X1, Y1, test_size = 0.2)

modele = LinearRegression()
modele.fit(X1_train, y1_train)

test1 = modele.predict(X1_test)

colonnes_utiles = ["P1_reel", "P2", "Lt", "Lq_reel", "P4", "P5", "V", "E2", "P6_reel", "E1", 'E2.2']
masque = df_normalize.columns.isin(colonnes_utiles)
df2 = df_normalize.loc[:, masque]

df_normalizedrop = df2.dropna()
df_normalizedrop.head()
print(len(df_normalizedrop))

X2 = df_normalizedrop.drop(columns = ['E2.2'])
Y2 = df_normalizedrop['E2.2']

X2_train, X2_test, y2_train, y2_test = train_test_split(X2, Y2, test_size = 0.2)

modele = LinearRegression()
modele.fit(X2_train, y2_train)

test2 = modele.predict(X2_test)

colonnes_utiles = ["P1_reel", "P2", "Lt", "Lq_reel", "P4", "P5", "V", "E2", "P6_reel", "E1", 'E2.3']
masque = df_normalize.columns.isin(colonnes_utiles)
df2 = df_normalize.loc[:, masque]

df_normalizedrop = df2.dropna()
df_normalizedrop.head()
print(len(df_normalizedrop))

X3 = df_normalizedrop.drop(columns = ['E2.3'])
Y3 = df_normalizedrop['E2.3']

X3_train, X3_test, y3_train, y3_test = train_test_split(X3, Y3, test_size = 0.2)

modele = LinearRegression()
modele.fit(X3_train, y3_train)

test3 = modele.predict(X_test)

colonnes_utiles = ["P1_reel", "P2", "Lt", "Lq_reel", "P4", "P5", "V", "E2", "P6_reel", "E1", 'E2.4']
masque = df_normalize.columns.isin(colonnes_utiles)
df2 = df_normalize.loc[:, masque]

df_normalizedrop = df2.dropna()
df_normalizedrop.head()
print(len(df_normalizedrop))

X4 = df_normalizedrop.drop(columns = ['E2.4'])
Y4 = df_normalizedrop['E2.4']

X4_train, X4_test, y4_train, y4_test = train_test_split(X4, Y4, test_size = 0.2)

modele = LinearRegression()
modele.fit(X4_train, y4_train)

test4 = modele.predict(X4_test)

df = df.apply(pd.to_numeric, errors='coerce')
t1 = test1.astype(float)
t2 = test2.astype(float)
t3 = test3.astype(float)
t4 = test4.astype(float)
y1t = y1_test.astype(float)
y2t = y2_test.astype(float)
y3t = y3_test.astype(float)
y4t = y4_test.astype(float)

Moyenne_reelle = (y1t.values + y2t.values + y3t.values + y4t.values)/4
Moyenne = (t1 + t2 + t3 + t4) / 4

erreur = mean_absolute_error(Moyenne, Moyenne_reelle)

print(f"Le modèle se trompe en moyenne de : {erreur}")

278
278
278


ValueError: The feature names should match those that were passed during fit.
Feature names seen at fit time, yet now missing:
- E1


In [ ]:

#Régression linéaire N°2 en supprimant les lignes NA (calcul direct moyenne)

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

df_normalize = df_normalize.apply(pd.to_numeric, errors='coerce')

colonnes_a_moyenner = ['E2.1', 'E2.2', 'E2.3', 'E2.4']
df_normalize['E2_Moyenne'] = df_normalize[colonnes_a_moyenner].mean(axis=1)

colonnes_utiles = ["P1_reel", "P2", "Lt", "Lq_reel", "P4", "P5", "V", "E2", "P6_reel", "E1", "E2_Moyenne"]
df_reduit = df_normalize[colonnes_utiles]

df_propre = df_reduit.dropna()
print(f"Nombre de lignes conservées pour l'entraînement : {len(df_propre)}")

X = df_propre.drop(columns=['E2_Moyenne'])
Y = df_propre['E2_Moyenne']

X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

modele = LinearRegression()
modele.fit(X_train, y_train)

predictions = modele.predict(X_test)
erreur = mean_absolute_error(y_test, predictions)

print(f"Le modèle se trompe en moyenne de : {erreur}")

Nombre de lignes conservées pour l'entraînement : 215
Le modèle se trompe en moyenne de : 0.04680520906322092
